In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/ĐATN/data/processed/embeddings/text_features_taskb"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Kết quả sẽ được lưu tại: {OUTPUT_DIR}")


Mounted at /content/drive
Kết quả sẽ được lưu tại: /content/drive/MyDrive/ĐATN/data/processed/embeddings/text_features_taskb


## Bước 0b — Tải dataset FashionIQ từ Kaggle

Cần file `kaggle.json` (API token) — vào Kaggle → Account → **Create New API Token** để tải file này, rồi upload lên khi cell dưới yêu cầu.


In [ ]:
!pip install -q kaggle

from google.colab import files
import os

os.makedirs('/root/.kaggle', exist_ok=True)
print("Vui lòng upload file kaggle.json:")
uploaded = files.upload()  # chọn kaggle.json

!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json


Vui lòng upload file kaggle.json:


Saving Kaggle Settings.json to Kaggle Settings.json
mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
DATA_ROOT = "/content/fashioniq_data"
os.makedirs(DATA_ROOT, exist_ok=True)

!kaggle datasets download -d binhhuunguyen/fashioniq -p {DATA_ROOT} --unzip

print("\nCấu trúc thư mục sau khi giải nén:")
!find {DATA_ROOT} -maxdepth 3 -type d


Dataset URL: https://www.kaggle.com/datasets/binhhuunguyen/fashioniq
License(s): unknown
100% 632M/632M [00:33<00:00, 19.8MB/s]


Cấu trúc thư mục sau khi giải nén:
/content/fashioniq_data
/content/fashioniq_data/fashion-iq-metadata-master
/content/fashioniq_data/fashion-iq-metadata-master/fashion-iq-metadata-master
/content/fashioniq_data/fashion-iq-metadata-master/fashion-iq-metadata-master/image_url
/content/fashioniq_data/fashion-iq-master
/content/fashioniq_data/fashion-iq-master/fashion-iq-master
/content/fashioniq_data/fashion-iq-master/fashion-iq-master/transformer
/content/fashioniq_data/fashion-iq-master/fashion-iq-master/captions
/content/fashioniq_data/fashion-iq-master/fashion-iq-master/start_kit
/content/fashioniq_data/fashion-iq-master/fashion-iq-master/image_splits
/content/fashioniq_data/resized_images
/content/fashioniq_data/resized_images/resized_images


## Bước 1 — Định nghĩa đường dẫn & khảo sát ảnh thật vs file rác macOS

Theo cấu trúc đã khảo sát: `captions/` nằm trong `fashion-iq-master/fashion-iq-master/`, ảnh nằm trong `resized_images/resized_images/` (có lẫn file rác `._*.jpg`).


In [ ]:
CAPTIONS_DIR = f"{DATA_ROOT}/fashion-iq-master/fashion-iq-master/captions"
IMAGES_DIR = f"{DATA_ROOT}/resized_images/resized_images"

assert os.path.isdir(CAPTIONS_DIR), f"Không tìm thấy: {CAPTIONS_DIR} — kiểm tra lại cấu trúc thư mục ở cell trên"
assert os.path.isdir(IMAGES_DIR), f"Không tìm thấy: {IMAGES_DIR} — kiểm tra lại cấu trúc thư mục ở cell trên"

all_files = os.listdir(IMAGES_DIR)
real_images = {f for f in all_files if not f.startswith('._') and f.lower().endswith('.jpg')}
junk_files = [f for f in all_files if f.startswith('._')]

print(f"Tổng file trong thư mục ảnh: {len(all_files)}")
print(f"Ảnh thật (đã lọc rác macOS): {len(real_images)}")
print(f"File rác '._*': {len(junk_files)}")


Tổng file trong thư mục ảnh: 74386
Ảnh thật (đã lọc rác macOS): 74381
File rác '._*': 5


## Bước 2 — Đọc 6 file caption JSON (bỏ qua `test` vì không có `target`)

Gắn nhãn `category` và `split` ngay khi đọc, để giữ được thông tin này xuyên suốt pipeline (cần cho việc tính Recall@K theo từng category sau này).


In [ ]:
import json
import pandas as pd

CATEGORIES = ["dress", "shirt", "toptee"]
SPLITS = ["train", "val"]   # bỏ 'test': không có field 'target'

raw_rows = []
for category in CATEGORIES:
    for split in SPLITS:
        json_path = f"{CAPTIONS_DIR}/cap.{category}.{split}.json"
        with open(json_path, "r") as f:
            entries = json.load(f)
        for e in entries:
            raw_rows.append({
                "category": category,
                "split": split,
                "reference_image_id": e["candidate"],
                "target_image_id": e["target"],
                "caption_1": e["captions"][0],
                "caption_2": e["captions"][1],
            })

df_raw = pd.DataFrame(raw_rows)
print(f"Tổng số triplet đọc được (trước khi lọc): {len(df_raw)}")
print(df_raw.groupby(['category','split']).size())


Tổng số triplet đọc được (trước khi lọc): 24016
category  split
dress     train    5985
          val      2017
shirt     train    5988
          val      2038
toptee    train    6027
          val      1961
dtype: int64


## Bước 3 — Lọc bỏ triplet thiếu ảnh (do broken links)


In [ ]:
def has_valid_images(row):
    ref_file = row['reference_image_id'] + '.jpg'
    tgt_file = row['target_image_id'] + '.jpg'
    return (ref_file in real_images) and (tgt_file in real_images)

mask_valid = df_raw.apply(has_valid_images, axis=1)
df_valid = df_raw[mask_valid].reset_index(drop=True)

n_dropped = len(df_raw) - len(df_valid)
print(f"Số triplet bị loại vì thiếu ảnh: {n_dropped} ({n_dropped/len(df_raw)*100:.2f}%)")
print(f"Số triplet còn lại: {len(df_valid)}")


Số triplet bị loại vì thiếu ảnh: 0 (0.00%)
Số triplet còn lại: 24016


## Bước 4 — Ghép 2 caption thành 1 câu modifier duy nhất

2 caption là 2 người gán nhãn độc lập mô tả **cùng một** sự khác biệt (không phải 2 lựa chọn thay thế), nên convention chuẩn là nối lại bằng "and". Không lowercase/loại stopword thủ công vì Fashion-CLIP dùng BPE tokenizer đã học trên câu tự nhiên.


In [ ]:
df_valid['modification_text'] = (
    df_valid['caption_1'].str.strip() + " and " + df_valid['caption_2'].str.strip()
)

# Loại bỏ caption rỗng hoặc quá ngắn (< 2 từ) — noise từ crowdsourcing
word_count_check = df_valid['modification_text'].str.split().str.len()
mask_ok = word_count_check >= 2

n_dropped_short = (~mask_ok).sum()
df_clean = df_valid[mask_ok].reset_index(drop=True)

print(f"Số dòng bị loại vì caption quá ngắn/rỗng: {n_dropped_short}")
print(f"Số triplet sạch cuối cùng: {len(df_clean)}")
df_clean[['reference_image_id','target_image_id','modification_text']].head()


Số dòng bị loại vì caption quá ngắn/rỗng: 0
Số triplet sạch cuối cùng: 24016


,reference_image_id,target_image_id,modification_text
0,B003FGW7MK,B008BHCT58,is solid black with no sleeves and is black wi...
1,B008MTHLHQ,B00BZ8GPVO,is longer and is lighter and longer
2,B00EVKYJAC,B008KZR6WM,3/4 sleeve black and white dress and more top ...
3,B003U9VL4W,B00684UYEE,is patterned and has a halter neckline and is ...
4,B00BLY2L2E,B00EV1B9C2,is black with no sleeves and is longer


## Bước 5 — Kiểm tra độ dài câu trước khi tokenize

In [ ]:
df_clean['word_count'] = df_clean['modification_text'].str.split().str.len()
print(df_clean['word_count'].describe())

pct_long = (df_clean['word_count'] > 30).mean() * 100
print(f"\n% câu > 30 từ (nguy cơ bị truncate ở max_length=77 token): {pct_long:.2f}%")


count    24016.000000
mean        11.662059
std          3.007391
min          3.000000
25%         10.000000
50%         12.000000
75%         14.000000
max         29.000000
Name: word_count, dtype: float64

% câu > 30 từ (nguy cơ bị truncate ở max_length=77 token): 0.00%


## Bước 6 — Lưu bảng dữ liệu tổng hợp (traceability)

In [ ]:
MASTER_CSV_PATH = f"{OUTPUT_DIR}/fashioniq_taskb_master.csv"
df_clean.to_csv(MASTER_CSV_PATH, index=False)
print(f"Đã lưu bảng tổng hợp: {MASTER_CSV_PATH}")
print(f"Số dòng: {len(df_clean)} — thứ tự hàng ở đây PHẢI khớp với thứ tự các file .npy sinh ra ở bước sau.")


Đã lưu bảng tổng hợp: /content/drive/MyDrive/ĐATN/data/processed/embeddings/text_features_taskb/fashioniq_taskb_master.csv
Số dòng: 24016 — thứ tự hàng ở đây PHẢI khớp với thứ tự các file .npy sinh ra ở bước sau.


## Bước 7 — Load Fashion-CLIP Text Encoder

Lấy `last_hidden_state` (toàn bộ token) + `attention_mask`, KHÔNG lấy `pooler_output`, vì XAA cần chuỗi token đầy đủ để Cross-Attention.


In [ ]:
!pip install -q transformers torch

import torch
from transformers import CLIPTokenizer, CLIPTextModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = "patrickjohncyh/fashion-clip"
MAX_LENGTH = 77
BATCH_SIZE = 64

tokenizer = CLIPTokenizer.from_pretrained(model_name)
text_model = CLIPTextModel.from_pretrained(model_name).to(device)
text_model.eval()

print(f"Đã load Fashion-CLIP text encoder trên: {device}")


tokenizer_config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] CLIPTextModel LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
visual_projection.weight                                       | UNEXPECTED |  | 
vision_model.post_layernorm.bias                               | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.embeddings.class_embedding                        | UNEXPECTED |  | 
vision_model.encoder.la

Đã load Fashion-CLIP text encoder trên: cuda


## Bước 8 — Tokenize + Encode toàn bộ modification_text theo batch

In [ ]:
import numpy as np
import time

modification_texts = df_clean['modification_text'].tolist()

all_hidden_states = []
all_attention_masks = []

start_time = time.time()
with torch.no_grad():
    for i in range(0, len(modification_texts), BATCH_SIZE):
        batch_texts = modification_texts[i:i + BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)

        outputs = text_model(**inputs)

        all_hidden_states.append(outputs.last_hidden_state.cpu().numpy())
        all_attention_masks.append(inputs["attention_mask"].cpu().numpy())

        if i % 1000 == 0 and i > 0:
            print(f"Đã xử lý {i} / {len(modification_texts)}")

text_hidden_states = np.concatenate(all_hidden_states, axis=0)
text_attention_masks = np.concatenate(all_attention_masks, axis=0)

print(f"Hoàn tất trong {time.time()-start_time:.2f}s")
print(f"hidden_states shape: {text_hidden_states.shape}")   # [N, 77, 512]
print(f"attention_masks shape: {text_attention_masks.shape}") # [N, 77]


Đã xử lý 8000 / 24016
Đã xử lý 16000 / 24016
Đã xử lý 24000 / 24016
Hoàn tất trong 70.48s
hidden_states shape: (24016, 77, 512)
attention_masks shape: (24016, 77)


## Bước 9 — Lưu kết quả (đầy đủ traceability)

Lưu kèm `category`, `split`, `reference_image_id`, `target_image_id` theo **đúng thứ tự hàng** với `text_hidden_states` — bắt buộc để không bị lệch dữ liệu khi ghép với Image Encoder sau này.


In [ ]:
np.save(f"{OUTPUT_DIR}/fashionclip_text_hidden_states.npy", text_hidden_states)
np.save(f"{OUTPUT_DIR}/fashionclip_text_attention_masks.npy", text_attention_masks)
np.save(f"{OUTPUT_DIR}/category.npy", df_clean['category'].values)
np.save(f"{OUTPUT_DIR}/split.npy", df_clean['split'].values)
np.save(f"{OUTPUT_DIR}/reference_image_ids.npy", df_clean['reference_image_id'].values)
np.save(f"{OUTPUT_DIR}/target_image_ids.npy", df_clean['target_image_id'].values)

print("Đã lưu toàn bộ output vào:", OUTPUT_DIR)
for fname in os.listdir(OUTPUT_DIR):
    print(" -", fname)


Đã lưu toàn bộ output vào: /content/drive/MyDrive/ĐATN/data/processed/embeddings/text_features_taskb
 - fashioniq_taskb_master.csv
 - fashionclip_text_hidden_states.npy
 - fashionclip_text_attention_masks.npy
 - category.npy
 - split.npy
 - reference_image_ids.npy
 - target_image_ids.npy


## Bước 10 — Sanity check cuối cùng

In [ ]:
sample_idx = 0

print("--- KIỂM TRA MẪU ĐẦU TIÊN ---")
print(f"Category: {df_clean['category'][sample_idx]} | Split: {df_clean['split'][sample_idx]}")
print(f"Reference image ID: {df_clean['reference_image_id'][sample_idx]}")
print(f"Target image ID:    {df_clean['target_image_id'][sample_idx]}")
print(f"Modification text:  {df_clean['modification_text'][sample_idx]}")

hs = text_hidden_states[sample_idx]
mask = text_attention_masks[sample_idx]
print(f"\nHidden state shape (seq_len, hidden_dim): {hs.shape}")
print(f"Số token thực (không tính padding): {mask.sum()}")

assert len(df_clean) == text_hidden_states.shape[0] == text_attention_masks.shape[0], \
    "LỖI: số dòng CSV và số hàng .npy không khớp — kiểm tra lại pipeline!"
print("\n✅ Số dòng khớp nhau giữa CSV và mọi file .npy — sẵn sàng cho bước Image Encoder / XAA.")


--- KIỂM TRA MẪU ĐẦU TIÊN ---
Category: dress | Split: train
Reference image ID: B003FGW7MK
Target image ID:    B008BHCT58
Modification text:  is solid black with no sleeves and is black with straps

Hidden state shape (seq_len, hidden_dim): (77, 512)
Số token thực (không tính padding): 13

✅ Số dòng khớp nhau giữa CSV và mọi file .npy — sẵn sàng cho bước Image Encoder / XAA.
